# Exercício 3 — Bag-of-Words por frequência e Embeddings

**Nome:** Tiago Galhardo Avelar

Este notebook reimplementa o classificador de sentimentos da aula 2 de duas formas:

1. vetor Bag-of-Words (BoW) por frequência + camada linear **sem bias**;
2. índices de tokens + `torch.nn.Embedding` e soma dos embeddings;
3. inicializações, ordem dos mini-batches e treinamento sincronizados para demonstrar que as duas formulações são numericamente equivalentes.

A equivalência usada é:

$$
Wc = \sum_{j=1}^{L} W_{:,t_j},
$$

em que $c$ é o histograma dos tokens, $t_j$ é o índice do token na posição $j$ e cada linha da tabela de embeddings é uma coluna de $W$.


## 1. Imports, configuração determinística e hiperparâmetros

Para reduzir diferenças de arredondamento entre multiplicação matricial e soma de embeddings, os modelos usam `float64`. A execução fica em CPU e os `DataLoader`s usam um gerador explícito, sempre reiniciado com a mesma seed.


In [1]:
import os
import random
import re
import tarfile
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from bs4 import BeautifulSoup
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 123
VOCAB_SIZE = 10_000
MAX_TOKENS = 200
HIDDEN_SIZE = 128
BATCH_SIZE = 50
N_EPOCHS = 5
LEARNING_RATE = 1e-4
DTYPE = torch.float64
DEVICE = torch.device("cpu")


def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)


set_seeds()
print("Dispositivo:", DEVICE)


Dispositivo: cpu


## 2. Download e carregamento do IMDb

O download só é feito se a pasta `aclImdb` ainda não existir. A divisão treino/validação é feita uma única vez usando a seed definida acima.


In [2]:
DATA_DIR = Path("aclImdb")
ARCHIVE = Path("aclImdb_v1.tar.gz")
DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

if not DATA_DIR.exists():
    if not ARCHIVE.exists():
        print("Baixando o dataset IMDb...")
        urllib.request.urlretrieve(DATA_URL, ARCHIVE)
    print("Extraindo o dataset...")
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall(".")


def load_texts(folder):
    paths = sorted(Path(folder).glob("*.txt"))
    return [path.read_text(encoding="utf-8") for path in paths]


x_train_pos = load_texts(DATA_DIR / "train" / "pos")
x_train_neg = load_texts(DATA_DIR / "train" / "neg")
x_test_pos = load_texts(DATA_DIR / "test" / "pos")
x_test_neg = load_texts(DATA_DIR / "test" / "neg")

train_pairs = list(zip(
    x_train_pos + x_train_neg,
    [1] * len(x_train_pos) + [0] * len(x_train_neg),
))
random.Random(SEED).shuffle(train_pairs)

n_train = int(0.8 * len(train_pairs))
train_pairs, valid_pairs = train_pairs[:n_train], train_pairs[n_train:]
test_pairs = list(zip(
    x_test_pos + x_test_neg,
    [1] * len(x_test_pos) + [0] * len(x_test_neg),
))

x_train, y_train = map(list, zip(*train_pairs))
x_valid, y_valid = map(list, zip(*valid_pairs))
x_test, y_test = map(list, zip(*test_pairs))

print(f"Treino: {len(x_train):,}")
print(f"Validação: {len(x_valid):,}")
print(f"Teste: {len(x_test):,}")


Treino: 20,000
Validação: 5,000
Teste: 25,000


## 3. Tokenização e vocabulário

O índice `UNK_IDX` representa palavras fora do vocabulário. O índice `PAD_IDX` existe apenas na representação por embeddings. Assim, o BoW possui `N_TOKENS` entradas e a tabela de embeddings possui `N_TOKENS + 1` linhas.


In [3]:
def tokenizer(sentence):
    text = BeautifulSoup(sentence, "html.parser").get_text(" ")
    text = re.sub(r"\W+", " ", text)
    return [token for token in re.findall(r"\w+", text.lower()) if not token.isnumeric()]


token_counts = Counter()
for text in x_train:
    token_counts.update(tokenizer(text))

# Ordem estável: maior frequência e, em caso de empate, ordem lexicográfica.
most_common = sorted(token_counts.items(), key=lambda item: (-item[1], item[0]))[:VOCAB_SIZE]
stoi = {token: index for index, (token, _) in enumerate(most_common)}

UNK_IDX = len(stoi)
N_TOKENS = len(stoi) + 1
PAD_IDX = N_TOKENS


def encode(text):
    return [stoi.get(token, UNK_IDX) for token in tokenizer(text)][:MAX_TOKENS]


print(f"Vocabulário conhecido: {len(stoi):,}")
print("UNK_IDX:", UNK_IDX, "PAD_IDX:", PAD_IDX)


Vocabulário conhecido: 10,000
UNK_IDX: 10000 PAD_IDX: 10001


## 4. Dataset: as duas representações dos mesmos tokens

Em `bow_freq`, cada exemplo é um histograma de frequências absolutas. Em `emb`, o mesmo exemplo é uma sequência de índices preenchida com `PAD_IDX` até `MAX_TOKENS`. O truncamento ocorre antes das duas transformações, portanto ambos os modelos recebem exatamente os mesmos tokens.


In [4]:
class SentimentDataset(Dataset):
    _encoded_cache = {}

    def __init__(self, texts, labels, mode):
        if mode not in {"bow_freq", "emb"}:
            raise ValueError("mode deve ser 'bow_freq' ou 'emb'")
        self.texts = texts
        self.labels = labels
        self.mode = mode
        # As duas representações reutilizam exatamente a mesma tokenização.
        cache_key = id(texts)
        if cache_key not in self._encoded_cache:
            self._encoded_cache[cache_key] = tuple(tuple(encode(text)) for text in texts)
        self.token_ids = self._encoded_cache[cache_key]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        token_ids = self.token_ids[index]

        if self.mode == "bow_freq":
            # bincount produz a frequência de cada token; não há normalização.
            x = torch.bincount(
                torch.tensor(token_ids, dtype=torch.long),
                minlength=N_TOKENS,
            ).to(DTYPE)
        else:
            x = torch.full((MAX_TOKENS,), PAD_IDX, dtype=torch.long)
            if token_ids:
                x[:len(token_ids)] = torch.tensor(token_ids, dtype=torch.long)

        return x, torch.tensor(self.labels[index], dtype=torch.long), index


## 5. Modelos equivalentes

A primeira camada do BoW é `Linear(..., bias=False)`. No modelo de embeddings, `EmbeddingBag(mode="sum")` faria a soma diretamente, mas aqui usamos explicitamente `nn.Embedding` para evidenciar o conceito pedido no exercício.


In [5]:
class BowModel(nn.Module):
    def __init__(self, n_tokens, hidden_size):
        super().__init__()
        self.first = nn.Linear(n_tokens, hidden_size, bias=False, dtype=DTYPE)
        self.output = nn.Linear(hidden_size, 2, bias=True, dtype=DTYPE)

    def forward(self, x):
        hidden = torch.relu(self.first(x))
        return self.output(hidden)


class EmbeddingModel(nn.Module):
    def __init__(self, n_tokens, hidden_size, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding(
            n_tokens + 1,
            hidden_size,
            padding_idx=padding_idx,
            dtype=DTYPE,
        )
        self.output = nn.Linear(hidden_size, 2, bias=True, dtype=DTYPE)

    def forward(self, x):
        hidden = torch.relu(self.embedding(x).sum(dim=1))
        return self.output(hidden)


def make_equivalent_models():
    """Cria os dois modelos e copia todos os parâmetros correspondentes."""
    set_seeds()
    bow = BowModel(N_TOKENS, HIDDEN_SIZE)
    emb = EmbeddingModel(N_TOKENS, HIDDEN_SIZE, PAD_IDX)

    with torch.no_grad():
        # Linear: [hidden, token]; Embedding: [token, hidden].
        emb.embedding.weight[:N_TOKENS].copy_(bow.first.weight.T)
        emb.embedding.weight[PAD_IDX].zero_()
        emb.output.weight.copy_(bow.output.weight)
        emb.output.bias.copy_(bow.output.bias)

    return bow.to(DEVICE), emb.to(DEVICE)


model_bow, model_emb = make_equivalent_models()

assert model_bow.first.bias is None
assert torch.equal(model_bow.first.weight.T, model_emb.embedding.weight[:N_TOKENS])
assert torch.equal(model_emb.embedding.weight[PAD_IDX], torch.zeros(HIDDEN_SIZE, dtype=DTYPE))
assert torch.equal(model_bow.output.weight, model_emb.output.weight)
assert torch.equal(model_bow.output.bias, model_emb.output.bias)

print(model_bow)
print(model_emb)


BowModel(
  (first): Linear(in_features=10001, out_features=128, bias=False)
  (output): Linear(in_features=128, out_features=2, bias=True)
)
EmbeddingModel(
  (embedding): Embedding(10002, 128, padding_idx=10001)
  (output): Linear(in_features=128, out_features=2, bias=True)
)


## 6. Mesmos mini-batches e equivalência antes do treino

Cada `DataLoader` embaralhado recebe um objeto `torch.Generator` independente, mas inicializado com a mesma seed. Desse modo, ambos percorrem os exemplos na mesma ordem.


In [6]:
def make_loader(texts, labels, mode, shuffle, seed=SEED):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        SentimentDataset(texts, labels, mode),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
    )


bow_train_loader = make_loader(x_train, y_train, "bow_freq", shuffle=True)
emb_train_loader = make_loader(x_train, y_train, "emb", shuffle=True)
bow_valid_loader = make_loader(x_valid, y_valid, "bow_freq", shuffle=False)
emb_valid_loader = make_loader(x_valid, y_valid, "emb", shuffle=False)

# Compara a saída de um mesmo lote antes de qualquer atualização.
bow_x, bow_y, bow_i = next(iter(make_loader(x_train, y_train, "bow_freq", shuffle=False)))
emb_x, emb_y, emb_i = next(iter(make_loader(x_train, y_train, "emb", shuffle=False)))

with torch.no_grad():
    bow_logits = model_bow(bow_x.to(DEVICE))
    emb_logits = model_emb(emb_x.to(DEVICE))

assert torch.equal(bow_y, emb_y)
assert torch.equal(bow_i, emb_i)
assert torch.allclose(bow_logits, emb_logits, rtol=1e-12, atol=1e-12)
print("Maior diferença nos logits antes do treino:", (bow_logits - emb_logits).abs().max().item())


Maior diferença nos logits antes do treino: 5.551115123125783e-17


## 7. Treinamento sincronizado

Os modelos são atualizados lote a lote usando o mesmo alvo. Além de garantir os mesmos mini-batches, o código verifica a equivalência dos logits e das perdas em cada passo. O Adam mantém estados equivalentes: o estado da matriz de embeddings é a transposta do estado da primeira camada linear (com uma linha extra e inerte para padding).


In [7]:
criterion = nn.CrossEntropyLoss()


def evaluate_pair(bow_model, emb_model, bow_loader, emb_loader):
    bow_model.eval()
    emb_model.eval()
    bow_loss = emb_loss = 0.0
    bow_correct = emb_correct = 0
    n = 0

    with torch.no_grad():
        for (xb, yb, ib), (xe, ye, ie) in zip(bow_loader, emb_loader):
            assert torch.equal(yb, ye)
            assert torch.equal(ib, ie)
            xb, xe, y = xb.to(DEVICE), xe.to(DEVICE), yb.to(DEVICE)
            logits_bow = bow_model(xb)
            logits_emb = emb_model(xe)
            assert torch.allclose(logits_bow, logits_emb, rtol=1e-10, atol=1e-11)

            loss_bow = criterion(logits_bow, y)
            loss_emb = criterion(logits_emb, y)
            bow_loss += loss_bow.item() * len(y)
            emb_loss += loss_emb.item() * len(y)
            bow_correct += (logits_bow.argmax(1) == y).sum().item()
            emb_correct += (logits_emb.argmax(1) == y).sum().item()
            n += len(y)

    return {
        "bow_loss": bow_loss / n,
        "emb_loss": emb_loss / n,
        "bow_acc": bow_correct / n,
        "emb_acc": emb_correct / n,
    }


def train_pair(bow_model, emb_model, bow_loader, emb_loader, epochs=N_EPOCHS):
    optimizer_bow = torch.optim.Adam(bow_model.parameters(), lr=LEARNING_RATE)
    optimizer_emb = torch.optim.Adam(emb_model.parameters(), lr=LEARNING_RATE)
    history = []

    for epoch in range(epochs):
        # Recria os loaders para que cada época comece com geradores idênticos.
        bow_loader = make_loader(x_train, y_train, "bow_freq", shuffle=True, seed=SEED + epoch)
        emb_loader = make_loader(x_train, y_train, "emb", shuffle=True, seed=SEED + epoch)
        bow_model.train()
        emb_model.train()
        bow_loss_sum = emb_loss_sum = 0.0
        n = 0

        for (xb, yb, ib), (xe, ye, ie) in zip(bow_loader, emb_loader):
            # Índices iguais provam que os geradores produziram a mesma permutação.
            assert torch.equal(yb, ye)
            assert torch.equal(ib, ie)
            xb, xe, y = xb.to(DEVICE), xe.to(DEVICE), yb.to(DEVICE)

            optimizer_bow.zero_grad(set_to_none=True)
            optimizer_emb.zero_grad(set_to_none=True)

            logits_bow = bow_model(xb)
            logits_emb = emb_model(xe)
            assert torch.allclose(logits_bow, logits_emb, rtol=1e-10, atol=1e-11)

            loss_bow = criterion(logits_bow, y)
            loss_emb = criterion(logits_emb, y)
            assert torch.allclose(loss_bow, loss_emb, rtol=1e-12, atol=1e-12)

            loss_bow.backward()
            loss_emb.backward()
            optimizer_bow.step()
            optimizer_emb.step()

            bow_loss_sum += loss_bow.item() * len(y)
            emb_loss_sum += loss_emb.item() * len(y)
            n += len(y)

        valid = evaluate_pair(
            bow_model,
            emb_model,
            bow_valid_loader,
            emb_valid_loader,
        )
        row = {
            "epoch": epoch + 1,
            "bow_train_loss": bow_loss_sum / n,
            "emb_train_loss": emb_loss_sum / n,
            **valid,
        }
        history.append(row)
        print(row)

    return history


history = train_pair(
    model_bow,
    model_emb,
    bow_train_loader,
    emb_train_loader,
)


{'epoch': 1, 'bow_train_loss': 0.48387211422236925, 'emb_train_loss': 0.48387211422236925, 'bow_loss': 0.36211516719153986, 'emb_loss': 0.36211516719153986, 'bow_acc': 0.8594, 'emb_acc': 0.8594}


{'epoch': 2, 'bow_train_loss': 0.29418800324973143, 'emb_train_loss': 0.29418800324973143, 'bow_loss': 0.3148926390020705, 'emb_loss': 0.31489263900207054, 'bow_acc': 0.8708, 'emb_acc': 0.8708}


{'epoch': 3, 'bow_train_loss': 0.2320955009716791, 'emb_train_loss': 0.2320955009716791, 'bow_loss': 0.3072691868802422, 'emb_loss': 0.30726918688024224, 'bow_acc': 0.8738, 'emb_acc': 0.8738}


{'epoch': 4, 'bow_train_loss': 0.1940453113622183, 'emb_train_loss': 0.1940453113622183, 'bow_loss': 0.3182786345714544, 'emb_loss': 0.3182786345714544, 'bow_acc': 0.8682, 'emb_acc': 0.8682}


{'epoch': 5, 'bow_train_loss': 0.16565230440980444, 'emb_train_loss': 0.1656523044098044, 'bow_loss': 0.3240559640330171, 'emb_loss': 0.3240559640330171, 'bow_acc': 0.8686, 'emb_acc': 0.8686}


## 8. Teste final e comprovação numérica

`torch.equal` é usado onde a cópia deve ser bit a bit idêntica. Para resultados calculados por operações diferentes, usa-se `torch.allclose` com tolerâncias estritas, pois a ordem de somas em ponto flutuante pode variar.


In [8]:
bow_test_loader = make_loader(x_test, y_test, "bow_freq", shuffle=False)
emb_test_loader = make_loader(x_test, y_test, "emb", shuffle=False)

test_results = evaluate_pair(
    model_bow,
    model_emb,
    bow_test_loader,
    emb_test_loader,
)

assert np.allclose(
    [row["bow_train_loss"] for row in history],
    [row["emb_train_loss"] for row in history],
    rtol=1e-10,
    atol=1e-11,
)
assert np.allclose(
    [row["bow_loss"] for row in history],
    [row["emb_loss"] for row in history],
    rtol=1e-10,
    atol=1e-11,
)
assert [row["bow_acc"] for row in history] == [row["emb_acc"] for row in history]
assert np.isclose(test_results["bow_loss"], test_results["emb_loss"], rtol=1e-10, atol=1e-11)
assert test_results["bow_acc"] == test_results["emb_acc"]

print("Resultados no teste:", test_results)
print("SUCESSO: BoW por frequência e Embedding produziram resultados numericamente idênticos.")


Resultados no teste: {'bow_loss': 0.36507806636591866, 'emb_loss': 0.36507806636591866, 'bow_acc': 0.85124, 'emb_acc': 0.85124}
SUCESSO: BoW por frequência e Embedding produziram resultados numericamente idênticos.


## Conclusão

O modelo BoW multiplica explicitamente o histograma de frequências pela matriz de pesos. O modelo com embeddings busca, para cada ocorrência de token, a coluna correspondente dessa mesma matriz (armazenada transposta) e soma os vetores. Como a primeira camada não possui bias, o padding vale zero, as demais camadas começam com os mesmos parâmetros e os mini-batches aparecem na mesma ordem, os dois procedimentos representam a mesma função e produzem as mesmas métricas dentro da precisão numérica definida.
